# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/neelamobaid/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


### Lane 2 — Content Refresh / Opportunity Scoring

- **Grain:** One row represents one content page for one client on one reporting date. The combination of `report_date`, `client_hash_id`, and `content_hash_id` is unique in the March 2026 slice.
- **Tables:** `fact_content_daily_performance` for daily page performance, with `dim_content` and `dim_clients` used for page and client context where needed.
- **Time window:** March 1–31, 2026 is the development slice. The available history differs by client, so source availability flags must be checked before using measurements.
- **Target/proxy:** A historical review-candidate proxy based on a decline in recent performance compared with a previous period. It is a prioritisation signal, not a verified editorial judgement.
- **Deliberate exclusion:** Client and content IDs are kept for grouping and tracking, not used as predictive features.


In [ ]:
!git clone https://github.com/neelamobaid/flyrank-ml-internship.git
%cd /content/flyrank-ml-internship

print("=== Skills README ===")
!cat skills/README.md

print("\n=== Writing Data Contracts Skill ===")
!cat skills/writing-data-contracts/SKILL.md

print("\n=== FlyRank Data Skill ===")
!cat skills/flyrank/flyrank-data/SKILL.md

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 129, done.
remote: Counting objects: 100% (129/129), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 129 (delta 40), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (129/129), 1.86 MiB | 15.21 MiB/s, done.
Resolving deltas: 100% (40/40), done.
/content/flyrank-ml-internship
=== Skills README ===
# Skills — the router

This folder is a small library of **skills**: focused instruction files your AI assistant loads
one at a time. One skill per task keeps the assistant sharp — its context window is small, and
filling it with everything makes it worse at the one thing you need.

**How to use it (repo-reading agents — Claude Code, Cursor, Codex):** they find this file
automatically via `AGENTS.md` / `CLAUDE.md`. Just tell your assistant which task you're doing.

**Using a chat-only assistant (ChatGPT / Gemini in a browser)?** Open the skill file on GitHub,
copy its whole content, an

In [ ]:
print("=== Querying Big Datasets Skill ===")
!cat skills/querying-big-datasets/SKILL.md

=== Querying Big Datasets Skill ===
---
name: querying-big-datasets
description: Works with datasets far too big to download or load in pandas — SQL over remote Parquet with DuckDB, aggregate-then-model, iterate on samples. Use when a dataset has millions of rows, lives on a remote host (hf:// or s3), or a notebook runs out of memory or hits rate limits.
---

# Querying big datasets

The trick that makes 79 million rows feel small: **never bring the rows to you — send the
question to the rows.** Aggregate in SQL, bring back only the small answer, model on that.

## The pattern

```python
%pip -q install duckdb
import duckdb
con = duckdb.connect()
# remote Parquet reads like a local table — match the path to how the table ships:
# most tables are ONE flat file at the repo root:
REL = "read_parquet('hf://datasets/<org>/<name>/<table>.parquet')"
# partitioned tables are a directory of month folders:
REL = "read_parquet('hf://datasets/<org>/<name>/<table>/month=2026-03/*.parquet')"
# sever

In [ ]:
%pip -q install duckdb huggingface_hub

In [2]:
from google.colab import userdata
import duckdb

hf_token = userdata.get("HF_TOKEN")
con = duckdb.connect()

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

safe_token = hf_token.replace("'", "''")

con.execute(f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{safe_token}'
    )
""")

print("Hugging Face authentication configured.")

Hugging Face authentication configured.


In [ ]:
# Check that we can read the warehouse table metadata
warehouse = "hf://datasets/FlyRank/internship-warehouse"

con.execute(f"""
    SELECT *
    FROM read_parquet('{warehouse}/dim_clients.parquet')
    LIMIT 5
""").df()

,client_hash_id,is_active,has_gsc_access,has_ga4_access,access_profile,client_created_date,client_updated_date,gsc_data_start,ga4_data_start
0,client_04660893ae39614a,True,True,True,gsc_and_ga4,2026-04-15,2026-06-27,NaT,2026-05-22
1,client_05475c07ed21a83a,True,False,False,no_search_or_analytics_access,2026-04-01,2026-06-27,NaT,NaT
2,client_06d356715a8ff3b6,True,True,True,gsc_and_ga4,2026-03-23,2026-07-05,2026-04-10,2026-04-06
3,client_0797ff3a1fc9a6a5,True,False,False,no_search_or_analytics_access,2025-05-26,2026-06-27,2025-11-05,NaT
4,client_08a6a72ff48e62c0,True,True,False,gsc_only,2025-05-26,2026-06-27,2025-09-24,NaT


In [ ]:
warehouse = "hf://datasets/FlyRank/internship-warehouse"

march_path = f"{warehouse}/fact_content_daily_performance/month=2026-03/*.parquet"

con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date
    FROM read_parquet('{march_path}')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,earliest_date,latest_date
0,9841378,2026-03-01,2026-03-31


In [ ]:
result = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date
    FROM read_parquet('{march_path}')
""").fetchone()

print("Total rows:", result[0])
print("Earliest date:", result[1])
print("Latest date:", result[2])

Total rows: 9841378
Earliest date: 2026-03-01
Latest date: 2026-03-31


In [ ]:
con.execute(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{march_path}')
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [ ]:
con.execute(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS duplicate_count
    FROM read_parquet('{march_path}')
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,duplicate_count


In [ ]:
con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
              AND ga4_data_available IS TRUE
        ) AS both_available_rows
    FROM read_parquet('{march_path}')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows,both_available_rows
0,9841378,3611061,413966,364347


In [ ]:
# Inspect the content dimension
print("=== dim_content ===")
print(con.execute("""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
    )
""").df().to_string(index=False))

# Inspect the 90-day query table
print("\n=== fact_content_query_90d ===")
print(con.execute("""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet'
    )
""").df().to_string(index=False))

=== dim_content ===
               column_name column_type null  key default extra
            client_hash_id     VARCHAR  YES None    None  None
           content_hash_id     VARCHAR  YES None    None  None
           keyword_hash_id     VARCHAR  YES None    None  None
               url_hash_id     VARCHAR  YES None    None  None
        keyword_char_count      BIGINT  YES None    None  None
       keyword_token_count      BIGINT  YES None    None  None
            url_char_count      BIGINT  YES None    None  None
      content_created_date        DATE  YES None    None  None
      content_updated_date        DATE  YES None    None  None
              content_type     VARCHAR  YES None    None  None
             search_volume      BIGINT  YES None    None  None
               competition      DOUBLE  YES None    None  None
         competition_level     VARCHAR  YES None    None  None
                       cpc      DOUBLE  YES None    None  None
               main_intent     VARC

In [3]:
query_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_query_90d.parquet"
)

con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(window_start) AS earliest_window_start,
        MAX(window_start) AS latest_window_start,
        MIN(window_end) AS earliest_window_end,
        MAX(window_end) AS latest_window_end,
        COUNT(DISTINCT window_start) AS distinct_window_starts,
        COUNT(DISTINCT window_end) AS distinct_window_ends
    FROM read_parquet('{query_path}')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,earliest_window_start,latest_window_start,earliest_window_end,latest_window_end,distinct_window_starts,distinct_window_ends
0,2414248,2026-04-02,2026-04-02,2026-06-30,2026-06-30,1,1


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.